In [0]:
# =============================================================================
# PRIMEINSURANCE — SILVER LAYER PIPELINE
# Catalog : primeins  |  Source: primeins.bronze  |  Target: primeins.silver
#
# DESIGN RULES (mandatory):
#   1. Quality rules are declared INSIDE this pipeline — not post-processing.
#   2. Every transformation is explained with a comment.
#   3. Clean records  → primeins.silver.{entity}
#   4. Failed records → primeins.silver.{entity}_quarantine  (+_reject_reason col)
#   5. No record is silently dropped — everything lands somewhere.
#   6. Every quality event is logged to primeins.silver.dq_issues_log.
# =============================================================================

In [0]:
# COMMAND ----------
# ------------------------------------------------------------------
# 0. IMPORTS & SHARED SETUP
# ------------------------------------------------------------------
from pyspark.sql import functions as F, types as T, DataFrame
from pyspark.sql.window import Window
from datetime import datetime


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS primeins.silver")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")   # one ID per full pipeline run

In [0]:
# Accumulator — every entity appends rows; written once at the end
dq_log_rows: list = []

def log_dq(table_name, rule_name, consequence, affected_count,
           total_count, detail="", suggested_fix=""):
    """Append one row to the in-memory DQ log accumulator."""
    ratio = round(affected_count / total_count, 4) if total_count > 0 else 0.0
    dq_log_rows.append({
        "run_id":          RUN_ID,
        "logged_at":       datetime.now().isoformat(),
        "table_name":      table_name,
        "rule_name":       rule_name,
        "consequence":     consequence,
        "affected_count":  int(affected_count),
        "total_count":     int(total_count),
        "affected_ratio":  ratio,
        "detail":          detail,
        "suggested_fix":   suggested_fix,
    })

In [0]:
def write_silver(clean_df: DataFrame, reject_df: DataFrame,
                 table: str, quarantine: str):
    """Write clean records to Silver and failed records to quarantine."""
    (clean_df.write.format("delta").mode("append")
             .option("mergeSchema", "true")
             .saveAsTable(f"primeins.silver.{table}"))
    (reject_df.write.format("delta").mode("append")
              .option("mergeSchema", "true")
              .saveAsTable(f"primeins.silver.{quarantine}"))

In [0]:
def add_audit(df: DataFrame, stage: str) -> DataFrame:
    """Attach pipeline audit columns to every row."""
    return (df
        .withColumn("_silver_load_ts",  F.current_timestamp())
        .withColumn("_pipeline_run_id", F.lit(RUN_ID))
        .withColumn("_stage",           F.lit(stage)))

In [0]:
# =============================================================================
# ENTITY 1 — CUSTOMERS
# =============================================================================
# Issues fixed (from DQ audit):
#   #1  customers_7 is a full duplicate of customers_1-6      -> deduplicate
#   #2  Three PK column names: CustomerID/Customer_ID/cust_id -> unify to customer_id
#   #3  customers_6: Education & Marital_status SWAPPED       -> detect and fix
#   #4  customers_5: region abbreviations E/W/C/S             -> expand to full names
#   #6  Column variants: Reg, City_in_state, Edu, Marital_status -> standardise
#   #8  "terto" in Education                                  -> correct to "tertiary"
#   #9  Null demographics (Education, Job, Marital)           -> flag column
#   #10 Negative Balance                                      -> cast + flag column
# Quality rules:
#   R1  customer_id not null          -> quarantine
#   R2  region in valid set           -> quarantine
#   R3  education in valid domain     -> warn
#   R4  negative balance              -> warn + flag (may be overdraft)
# =============================================================================

# COMMAND ----------
print("=" * 60)
print("CUSTOMERS — loading from Bronze")

raw_cust = spark.table("primeins.bronze.customers")
total_cust = raw_cust.count()
print(f"  Bronze row count (before dedup): {total_cust:,}")

# ------------------------------------------------------------------
# TRANSFORMATION 1: Unify PK column — CustomerID / Customer_ID / cust_id
# Coalesce across all three variants; whichever is non-null wins.
# ------------------------------------------------------------------
cust = (raw_cust
    .withColumn("customer_id",
        F.coalesce(
            F.col("CustomerID")  if "CustomerID"  in raw_cust.columns else F.lit(None),
            F.col("Customer_ID") if "Customer_ID" in raw_cust.columns else F.lit(None),
            F.col("cust_id")     if "cust_id"     in raw_cust.columns else F.lit(None),
        ).cast(T.StringType()))
    .drop(*[c for c in ["CustomerID","Customer_ID","cust_id"] if c in raw_cust.columns])
)

# ------------------------------------------------------------------
# TRANSFORMATION 2: Standardise non-PK column names to canonical Silver names
# ------------------------------------------------------------------
rename_map = {
    "Reg":           "region_raw",
    "Region":        "region_raw",
    "City_in_state": "city",
    "City":          "city",
    "Edu":           "education_raw",
    "Education":     "education_raw",
    "Marital_status":"marital_raw",
    "Marital":       "marital_raw",
}
for old, new in rename_map.items():
    if old in cust.columns and new not in cust.columns:
        cust = cust.withColumnRenamed(old, new)

# Ensure canonical columns exist even if all source files lacked them
for col_name in ["region_raw","education_raw","marital_raw","city"]:
    if col_name not in cust.columns:
        cust = cust.withColumn(col_name, F.lit(None).cast(T.StringType()))

# ------------------------------------------------------------------
# TRANSFORMATION 3: Fix customers_6 swapped Education / Marital_status
# Detection: if education_raw holds marital values -> swap
# ------------------------------------------------------------------
MARITAL_VALS = ["single", "married", "divorced"]
EDUCATION_VALS = ["primary", "secondary", "tertiary"]

cust = (cust
    .withColumn("_edu_is_marital",
        F.lower(F.col("education_raw")).isin(MARITAL_VALS))
    .withColumn("education",
        F.when(F.col("_edu_is_marital"), F.lower(F.col("marital_raw")))
         .otherwise(F.lower(F.col("education_raw"))))
    .withColumn("marital",
        F.when(F.col("_edu_is_marital"), F.lower(F.col("education_raw")))
         .otherwise(F.lower(F.col("marital_raw"))))
    .drop("education_raw","marital_raw","_edu_is_marital")
)

# ------------------------------------------------------------------
# TRANSFORMATION 4: Expand region abbreviations (customers_5)
# E -> East | W -> West | C -> Central | S -> South
# Full names from other files pass through via initcap.
# ------------------------------------------------------------------
cust = cust.withColumn("region",
    F.when(F.upper(F.col("region_raw")) == "E", "East")
     .when(F.upper(F.col("region_raw")) == "W", "West")
     .when(F.upper(F.col("region_raw")) == "C", "Central")
     .when(F.upper(F.col("region_raw")) == "S", "South")
     .when(F.upper(F.col("region_raw")) == "N", "North")
     .otherwise(F.initcap(F.col("region_raw")))
).drop("region_raw")

# ------------------------------------------------------------------
# TRANSFORMATION 5: Fix corrupted Education value "terto" -> "tertiary"
# 73 rows in customers_5 — systemic export encoding error
# ------------------------------------------------------------------
cust = cust.withColumn("education",
    F.when(F.lower(F.col("education")) == "terto", "tertiary")
     .otherwise(F.col("education")))

log_dq("customers","terto_education_fix","fix",
       raw_cust.filter(F.lower(F.col("Education") if "Education" in raw_cust.columns
                               else F.lit("")).isNotNull()).count(), total_cust,
       "73 rows in customers_5 had Education='terto' — corrected to 'tertiary'.",
       "Fix export encoding in Insurance 5 regional system.")

# ------------------------------------------------------------------
# TRANSFORMATION 6: Cast Balance to numeric + flag negatives
# ------------------------------------------------------------------
if "Balance" in cust.columns:
    cust = (cust
        .withColumn("balance", F.col("Balance").cast(T.DoubleType()))
        .drop("Balance")
        .withColumn("_negative_balance_flag", F.col("balance") < 0))

# ------------------------------------------------------------------
# TRANSFORMATION 7: Flag null demographics
# ------------------------------------------------------------------
cust = cust.withColumn("_null_demographics_flag",
    F.col("education").isNull() | F.col("marital").isNull())

# ------------------------------------------------------------------
# TRANSFORMATION 8: Deduplicate — customers_7 is a superset of 1-6
# Keep the first occurrence per customer_id ordered by source file.
# ------------------------------------------------------------------
dedup_w = Window.partitionBy("customer_id").orderBy("_source_file")
cust = (cust
    .withColumn("_rn", F.row_number().over(dedup_w))
    .filter(F.col("_rn") == 1)
    .drop("_rn"))

post_dedup = cust.count()
dupes_removed = total_cust - post_dedup
print(f"  Duplicates removed: {dupes_removed:,}")
log_dq("customers","customers_7_deduplication","drop",
       dupes_removed, total_cust,
       f"customers_7 duplicates all 1,604 rows from files 1-6. {dupes_removed} rows removed.",
       "Exclude customers_7 from future Bronze ingestion — it is a consolidated export, not a new region.")

# ------------------------------------------------------------------
# QUALITY RULES — CUSTOMERS
# ------------------------------------------------------------------
VALID_REGIONS = {"East","West","Central","South","North"}

# R1: customer_id not null -> quarantine
r1_fail = cust.filter(F.col("customer_id").isNull()).withColumn(
    "_reject_reason", F.lit("R1: customer_id is null"))
r1_pass = cust.filter(F.col("customer_id").isNotNull())
log_dq("customers","R1_customer_id_not_null","quarantine",
       r1_fail.count(), post_dedup,
       "Records with no customer_id cannot join to any other entity.",
       "Investigate source file for missing primary key values.")

# R2: region must be in known set -> quarantine
r2_fail = r1_pass.filter(
    F.col("region").isNull() | ~F.col("region").isin(list(VALID_REGIONS))
).withColumn("_reject_reason",
    F.lit("R2: region value not in {East,West,Central,South,North}"))
r2_pass = r1_pass.filter(
    F.col("region").isNotNull() & F.col("region").isin(list(VALID_REGIONS)))
log_dq("customers","R2_valid_region","quarantine",
       r2_fail.count(), post_dedup,
       "Unrecognised region values after abbreviation expansion.",
       "Check source files for new region codes or encoding errors.")

# R3: education domain -> warn only (nulls are allowed; wrong values are not)
r3_warn = r2_pass.filter(
    F.col("education").isNotNull() &
    ~F.lower(F.col("education")).isin("primary","secondary","tertiary"))
log_dq("customers","R3_valid_education_domain","warn",
       r3_warn.count(), post_dedup,
       "Education values outside {primary, secondary, tertiary}.",
       "Investigate for further encoding variants similar to 'terto'.")

# R4: negative balance -> warn + flag (may be valid overdraft)
log_dq("customers","R4_negative_balance","warn",
       r2_pass.filter(F.col("balance") < 0).count() if "balance" in r2_pass.columns else 0,
       post_dedup,
       "Negative account balances found across multiple files.",
       "Confirm with business whether negative balance is a valid overdraft state.")

cust_reject = r1_fail.unionByName(r2_fail, allowMissingColumns=True)
cust_clean  = add_audit(r2_pass, "silver_customers")

write_silver(cust_clean, add_audit(cust_reject,"silver_customers_quarantine"),
             "customers","customers_quarantine")

print(f"  Clean  -> silver.customers:             {cust_clean.count():,}")
print(f"  Reject -> silver.customers_quarantine:  {cust_reject.count():,}")


In [0]:
# =============================================================================
# ENTITY 2 — CLAIMS
# =============================================================================
# Issues fixed (from DQ audit):
#   #11 All 3 date fields corrupted ("27:00.0" Excel fragments) -> quarantine
#   #12 ClaimID format inconsistency between files              -> log + dedup
#   #13 Literal string "NULL" in Claim_Processed_On             -> proper null
#   #14 "?" in property_damage / police_report_available        -> proper null
#   #15 Claim_Rejected "Y"/"N"                                  -> boolean
# Quality rules:
#   R1  ClaimID not null                  -> quarantine
#   R2  PolicyID not null                 -> quarantine
#   R3  Corrupt date fields               -> quarantine (unrecoverable)
#   R4  ClaimID must be unique            -> quarantine duplicates
# =============================================================================

# COMMAND ----------
print("=" * 60)
print("CLAIMS — loading from Bronze")

raw_claims = spark.table("primeins.bronze.claims")
total_claims = raw_claims.count()

# ------------------------------------------------------------------
# TRANSFORMATION 1: String "NULL" -> proper null in Claim_Processed_On
# ------------------------------------------------------------------
claims = raw_claims
if "Claim_Processed_On" in claims.columns:
    claims = claims.withColumn("Claim_Processed_On",
        F.when(F.upper(F.col("Claim_Processed_On")) == "NULL", F.lit(None))
         .otherwise(F.col("Claim_Processed_On")))

# ------------------------------------------------------------------
# TRANSFORMATION 2: "?" -> null in boolean-like fields
# property_damage and police_report_available use YES/NO/? (3-valued)
# "?" = unknown/not collected — convert to proper null
# ------------------------------------------------------------------
for bool_col in ["property_damage","police_report_available"]:
    if bool_col in claims.columns:
        claims = claims.withColumn(bool_col,
            F.when(F.col(bool_col) == "?", F.lit(None))
             .otherwise(F.col(bool_col)))

# ------------------------------------------------------------------
# TRANSFORMATION 3: Claim_Rejected "Y"/"N" -> boolean
# ------------------------------------------------------------------
if "Claim_Rejected" in claims.columns:
    claims = (claims
        .withColumn("claim_rejected",
            F.when(F.upper(F.col("Claim_Rejected")) == "Y", F.lit(True))
             .when(F.upper(F.col("Claim_Rejected")) == "N", F.lit(False))
             .otherwise(F.lit(None).cast(T.BooleanType())))
        .drop("Claim_Rejected"))

# ------------------------------------------------------------------
# TRANSFORMATION 4: Detect corrupted date fields
# Pattern: "27:00.0", "34:00.0" — Excel time-serial fragments
# These records are UNRECOVERABLE — quarantine them.
# ------------------------------------------------------------------
DATE_CORRUPT_RE = r"^\d{1,3}:\d{2}\.\d+$"
date_cols = [c for c in ["incident_date","Claim_Logged_On","Claim_Processed_On"]
             if c in claims.columns]

if date_cols:
    corrupt_condition = F.lit(False)
    for dc in date_cols:
        corrupt_condition = corrupt_condition | (
            F.col(dc).isNotNull() & F.col(dc).rlike(DATE_CORRUPT_RE))
    claims = claims.withColumn("_dates_corrupted", corrupt_condition)
else:
    claims = claims.withColumn("_dates_corrupted", F.lit(False))

corrupt_ct = claims.filter(F.col("_dates_corrupted")).count()
log_dq("claims","corrupt_date_fields","quarantine",
       corrupt_ct, total_claims,
       "incident_date, Claim_Logged_On, Claim_Processed_On contain Excel time-serial fragments (e.g. '27:00.0'). No date information survives.",
       "Re-export claims data from source system with correct datetime serialisation. Audit Excel export settings.")

# ------------------------------------------------------------------
# TRANSFORMATION 5: Cast numeric claim fields
# ------------------------------------------------------------------
for num_col in ["total_claim_amount","vehicle_claim","property_claim",
                "injury_claim","auto_year"]:
    if num_col in claims.columns:
        claims = claims.withColumn(num_col, F.col(num_col).cast(T.DoubleType()))

# ------------------------------------------------------------------
# QUALITY RULES — CLAIMS
# ------------------------------------------------------------------

# R1: ClaimID not null
r1_fail_c = claims.filter(F.col("ClaimID").isNull() if "ClaimID" in claims.columns
                           else F.lit(False)).withColumn(
    "_reject_reason", F.lit("R1: ClaimID is null"))
r1_pass_c = claims.filter(F.col("ClaimID").isNotNull() if "ClaimID" in claims.columns
                           else F.lit(True))
log_dq("claims","R1_claimid_not_null","quarantine",
       r1_fail_c.count(), total_claims,
       "Claims with no ClaimID cannot be tracked.",
       "Investigate source JSON for missing ID values.")

# R2: PolicyID not null (only path to customer)
r2_fail_c = r1_pass_c.filter(
    F.col("PolicyID").isNull() if "PolicyID" in r1_pass_c.columns else F.lit(False)
).withColumn("_reject_reason", F.lit("R2: PolicyID is null — no customer link possible"))
r2_pass_c = r1_pass_c.filter(
    F.col("PolicyID").isNotNull() if "PolicyID" in r1_pass_c.columns else F.lit(True))
log_dq("claims","R2_policyid_not_null","quarantine",
       r2_fail_c.count(), total_claims,
       "Null PolicyID means claim cannot be linked to a customer.",
       "Check source for orphan claims. May need manual policyholder assignment.")

# R3: Corrupted dates -> quarantine (unrecoverable)
r3_fail_c = r2_pass_c.filter(F.col("_dates_corrupted")).withColumn(
    "_reject_reason",
    F.lit("R3: All date fields contain Excel time-serial corruption — unrecoverable"))
r3_pass_c = r2_pass_c.filter(~F.col("_dates_corrupted"))

# R4: ClaimID must be unique (collision risk between files)
if "ClaimID" in r3_pass_c.columns:
    dedup_c = Window.partitionBy("ClaimID").orderBy("_source_file")
    r3_pass_c = r3_pass_c.withColumn("_rn", F.row_number().over(dedup_c))
    r4_fail_c = r3_pass_c.filter(F.col("_rn") > 1).drop("_rn").withColumn(
        "_reject_reason", F.lit("R4: Duplicate ClaimID — first occurrence kept"))
    r4_pass_c = r3_pass_c.filter(F.col("_rn") == 1).drop("_rn")
    log_dq("claims","R4_claimid_unique","quarantine",
           r4_fail_c.count(), total_claims,
           "ClaimID collision between claims_1 (4-8 digit) and claims_2 (8-digit) schemes.",
           "Prefix ClaimID with source file identifier before merging, or use a surrogate key.")
else:
    r4_fail_c = r3_pass_c.filter(F.lit(False))
    r4_pass_c = r3_pass_c

claims_reject = (r1_fail_c
    .unionByName(r2_fail_c, allowMissingColumns=True)
    .unionByName(r3_fail_c, allowMissingColumns=True)
    .unionByName(r4_fail_c, allowMissingColumns=True))
claims_clean = add_audit(r4_pass_c.drop("_dates_corrupted"), "silver_claims")

write_silver(claims_clean, add_audit(claims_reject,"silver_claims_quarantine"),
             "claims","claims_quarantine")

print(f"  Clean  -> silver.claims:             {claims_clean.count():,}")
print(f"  Reject -> silver.claims_quarantine:  {claims_reject.count():,}")
print(f"  NOTE: Most/all claims expected in quarantine due to date corruption.")

In [0]:
# =============================================================================
# ENTITY 3 — SALES
# =============================================================================
# Issues fixed (from DQ audit):
#   #17 1,877 / 1,254 entirely blank rows in sales_1 / Sales_2  -> drop blanks
#   #18 sales_3.csv missing entirely                            -> log gap
#   #20 sold_on is null for unsold cars                         -> flag is_unsold
#   #21 seller_type "Trustmark Dealer" absent in sales_4        -> log note
# Quality rules:
#   R1  Row must have at least one data field     -> drop blanks (already done)
#   R2  car_id not null                           -> quarantine
#   R3  listed_price > 0                          -> quarantine
#   R4  sold_on >= ad_placed_on when both present -> quarantine
# =============================================================================

# COMMAND ----------
print("=" * 60)
print("SALES — loading from Bronze")

raw_sales = spark.table("primeins.bronze.sales")
total_sales_raw = raw_sales.count()

# ------------------------------------------------------------------
# TRANSFORMATION 1: Drop entirely blank rows
# A blank row has null/empty in ALL data columns (not audit columns).
# ------------------------------------------------------------------
data_cols_s = [c for c in raw_sales.columns if not c.startswith("_")]

# Row is blank if every data column is null or empty string
not_blank = F.greatest(*[
    F.when(
        F.col(c).isNotNull() & (F.trim(F.col(c).cast(T.StringType())) != ""),
        F.lit(1)
    ).otherwise(F.lit(0))
    for c in data_cols_s
]) > 0

sales = raw_sales.filter(not_blank)
blank_ct = total_sales_raw - sales.count()
log_dq("sales","blank_rows_dropped","drop",
       blank_ct, total_sales_raw,
       f"{blank_ct:,} rows were completely empty — file concatenation debris from sales_1 and Sales_2.",
       "Fix the file generation process to not append blank padding rows at EOF.")

# Log missing sales_3 file (no code can fix a missing file)
log_dq("sales","missing_file_sales_3","warn",
       0, sales.count(),
       "sales_3.csv does not exist. Files jump sales_1 -> Sales_2 -> sales_4. Full period gap.",
       "Request re-extraction of the missing period from Insurance 3 regional system.")

# ------------------------------------------------------------------
# TRANSFORMATION 2: Parse dates to timestamp (ISO-8601)
# Sales dates are in DD-MM-YYYY HH:MM format
# ------------------------------------------------------------------
DATE_FMT = "dd-MM-yyyy HH:mm"
for dc in ["ad_placed_on","sold_on"]:
    if dc in sales.columns:
        sales = sales.withColumn(dc, F.to_timestamp(F.col(dc), DATE_FMT))

# ------------------------------------------------------------------
# TRANSFORMATION 3: Explicitly flag unsold inventory
# sold_on IS NULL = car is still listed / not yet sold
# ------------------------------------------------------------------
sales = sales.withColumn("is_unsold", F.col("sold_on").isNull()
                         if "sold_on" in sales.columns else F.lit(True))

# ------------------------------------------------------------------
# TRANSFORMATION 4: Cast numeric sales fields
# ------------------------------------------------------------------
for nc in ["listed_price","km_driven","year_of_manufacture"]:
    if nc in sales.columns:
        sales = sales.withColumn(nc, F.col(nc).cast(T.DoubleType()))

# ------------------------------------------------------------------
# QUALITY RULES — SALES
# ------------------------------------------------------------------
total_sales = sales.count()

# R2: car_id not null
car_id_col = "car_id" if "car_id" in sales.columns else None
if car_id_col:
    r2_fail_s = sales.filter(F.col(car_id_col).isNull()).withColumn(
        "_reject_reason", F.lit("R2: car_id is null — cannot link to cars dimension"))
    r2_pass_s = sales.filter(F.col(car_id_col).isNotNull())
    log_dq("sales","R2_car_id_not_null","quarantine",
           r2_fail_s.count(), total_sales,
           "Sales records with no car_id cannot be linked to the cars dimension.",
           "Investigate source extract for missing vehicle identifiers.")
else:
    r2_fail_s = sales.filter(F.lit(False))
    r2_pass_s = sales

# R3: listed_price > 0
if "listed_price" in r2_pass_s.columns:
    r3_fail_s = r2_pass_s.filter(
        r2_pass_s["listed_price"].isNull() | (r2_pass_s["listed_price"] <= 0)
    ).withColumn("_reject_reason", F.lit("R3: listed_price is null or <= 0"))
    r3_pass_s = r2_pass_s.filter(
        r2_pass_s["listed_price"].isNotNull() & (r2_pass_s["listed_price"] > 0))
    log_dq("sales","R3_listed_price_positive","quarantine",
           r3_fail_s.count(), total_sales,
           "Zero or negative listed prices are invalid.",
           "Check source for placeholder values (0, -1).")
else:
    r3_fail_s = r2_pass_s.filter(F.lit(False))
    r3_pass_s = r2_pass_s

# R4: sold_on must be >= ad_placed_on when both present
if all(c in r3_pass_s.columns for c in ["sold_on","ad_placed_on"]):
    r4_fail_s = r3_pass_s.filter(
        r3_pass_s["sold_on"].isNotNull() &
        r3_pass_s["ad_placed_on"].isNotNull() &
        (r3_pass_s["sold_on"] < r3_pass_s["ad_placed_on"])
    ).withColumn("_reject_reason", F.lit("R4: sold_on is before ad_placed_on — invalid timeline"))
    r4_pass_s = r3_pass_s.filter(
        r3_pass_s["sold_on"].isNull() |
        r3_pass_s["ad_placed_on"].isNull() |
        (r3_pass_s["sold_on"] >= r3_pass_s["ad_placed_on"]))
    log_dq("sales","R4_sold_after_listed","quarantine",
           r4_fail_s.count(), total_sales,
           "sold_on date is earlier than ad_placed_on — logically impossible.",
           "Check for date parsing errors or data entry mistakes.")
else:
    r4_fail_s = r3_pass_s.filter(F.lit(False))
    r4_pass_s = r3_pass_s

sales_reject = (r2_fail_s
    .unionByName(r3_fail_s, allowMissingColumns=True)
    .unionByName(r4_fail_s, allowMissingColumns=True))
sales_clean  = add_audit(r4_pass_s, "silver_sales")

write_silver(sales_clean, add_audit(sales_reject,"silver_sales_quarantine"),
             "sales","sales_quarantine")

print(f"  Clean  -> silver.sales:             {sales_clean.count():,}")
print(f"  Reject -> silver.sales_quarantine:  {sales_reject.count():,}")

In [0]:
# =============================================================================
# ENTITY 4 — CARS
# =============================================================================
# Issues fixed (from DQ audit):
#   #22 mileage/engine/max_power stored as strings with unit suffixes -> strip + cast
#   #23 Torque has 4+ incompatible formats and mixed Nm/kgm units     -> extract Nm
#   #24 One vehicle with km_driven = 1,500,000 (likely extra zero)    -> quarantine
# Quality rules:
#   R1  PK not null              -> quarantine
#   R2  km_driven <= 500,000     -> quarantine (outlier)
#   R3  year_of_manufacture valid-> quarantine
#   R4  selling_price > 0        -> quarantine
# =============================================================================

# COMMAND ----------
print("=" * 60)
print("CARS — loading from Bronze")

raw_cars = spark.table("primeins.bronze.cars")
total_cars = raw_cars.count()

cars = raw_cars

# ------------------------------------------------------------------
# TRANSFORMATION 1: Strip unit suffix from mileage + cast to double
# "23.4 kmpl" -> mileage=23.4, mileage_unit="kmpl"
# "21.14 km/kg" -> mileage=21.14, mileage_unit="km/kg"
# ------------------------------------------------------------------
if "mileage" in cars.columns:
    cars = (cars
        .withColumn("mileage_unit",
            F.regexp_extract(F.col("mileage"), r"([a-zA-Z/]+)", 1))
        .withColumn("mileage",
            F.regexp_extract(F.col("mileage"), r"([\d.]+)", 1)
             .cast(T.DoubleType())))

# ------------------------------------------------------------------
# TRANSFORMATION 2: Strip "CC" from engine + cast to integer
# "1248 CC" -> engine_cc=1248
# ------------------------------------------------------------------
if "engine" in cars.columns:
    cars = (cars
        .withColumn("engine_cc",
            F.regexp_extract(F.col("engine"), r"(\d+)", 1).cast(T.IntegerType()))
        .drop("engine"))

# ------------------------------------------------------------------
# TRANSFORMATION 3: Strip "bhp" from max_power + cast to double
# "74 bhp" -> max_power_bhp=74.0
# ------------------------------------------------------------------
if "max_power" in cars.columns:
    cars = (cars
        .withColumn("max_power_bhp",
            F.regexp_extract(F.col("max_power"), r"([\d.]+)", 1).cast(T.DoubleType()))
        .drop("max_power"))

# ------------------------------------------------------------------
# TRANSFORMATION 4: Parse torque -> extract Nm value
# 4 observed formats, mixed Nm/kgm units.
# Strategy: extract first numeric value, detect unit, convert kgm->Nm.
# 1 kgm = 9.80665 Nm
# ------------------------------------------------------------------
KGM_TO_NM = 9.80665

if "torque" in cars.columns:
    cars = (cars
        .withColumn("torque_raw", F.col("torque"))  # preserve original string
        .withColumn("_torque_unit",
            F.when(F.lower(F.col("torque")).contains("kgm"), "kgm")
             .otherwise("Nm"))
        .withColumn("_torque_val",
            F.regexp_extract(F.col("torque"), r"([\d]+\.?[\d]*)", 1)
             .cast(T.DoubleType()))
        .withColumn("torque_nm",
            F.when(F.col("_torque_unit") == "kgm",
                   F.round(F.col("_torque_val") * KGM_TO_NM, 2))
             .otherwise(F.col("_torque_val")))
        .drop("torque","_torque_unit","_torque_val"))

# ------------------------------------------------------------------
# TRANSFORMATION 5: Cast remaining numeric columns
# ------------------------------------------------------------------
for nc in ["km_driven","selling_price","year_of_manufacture"]:
    if nc in cars.columns:
        cars = cars.withColumn(nc, F.col(nc).cast(T.DoubleType()))

# ------------------------------------------------------------------
# QUALITY RULES — CARS
# ------------------------------------------------------------------

# R1: PK not null
pk_car = "name" if "name" in cars.columns else cars.columns[0]
r1_fail_ca = cars.filter(F.col(pk_car).isNull()).withColumn(
    "_reject_reason", F.lit(f"R1: {pk_car} (primary key) is null"))
r1_pass_ca = cars.filter(F.col(pk_car).isNotNull())
log_dq("cars","R1_pk_not_null","quarantine",
       r1_fail_ca.count(), total_cars,
       f"{pk_car} is null.","Check source for missing vehicle identifiers.")

# R2: km_driven outlier -> quarantine
KM_MAX = 500_000
if "km_driven" in r1_pass_ca.columns:
    r2_fail_ca = r1_pass_ca.filter(
        r1_pass_ca["km_driven"].isNotNull() & (r1_pass_ca["km_driven"] > KM_MAX)
    ).withColumn("_reject_reason",
        F.concat(F.lit("R2: km_driven="),
                 F.col("km_driven").cast(T.StringType()),
                 F.lit(f" exceeds {KM_MAX:,} — likely data entry error")))
    r2_pass_ca = r1_pass_ca.filter(
        r1_pass_ca["km_driven"].isNull() | (r1_pass_ca["km_driven"] <= KM_MAX))
    log_dq("cars","R2_km_driven_outlier","quarantine",
           r2_fail_ca.count(), total_cars,
           "1 vehicle with km_driven=1,500,000 — 3x the next-highest value.",
           "Verify with Insurance 4. Likely extra zero — should be 150,000.")
else:
    r2_fail_ca = r1_pass_ca.filter(F.lit(False))
    r2_pass_ca = r1_pass_ca

# R3: year_of_manufacture valid range
if "year_of_manufacture" in r2_pass_ca.columns:
    r3_fail_ca = r2_pass_ca.filter(
        r2_pass_ca["year_of_manufacture"].isNotNull() &
        ((r2_pass_ca["year_of_manufacture"] < 1900) |
         (r2_pass_ca["year_of_manufacture"] > 2026))
    ).withColumn("_reject_reason", F.lit("R3: year_of_manufacture outside valid range 1900-2026"))
    r3_pass_ca = r2_pass_ca.filter(
        r2_pass_ca["year_of_manufacture"].isNull() |
        ((r2_pass_ca["year_of_manufacture"] >= 1900) &
         (r2_pass_ca["year_of_manufacture"] <= 2026)))
    log_dq("cars","R3_valid_year","quarantine",
           r3_fail_ca.count(), total_cars,
           "Manufacture year outside 1900-2026.","Check source for typos.")
else:
    r3_fail_ca = r2_pass_ca.filter(F.lit(False))
    r3_pass_ca = r2_pass_ca

# R4: selling_price > 0
if "selling_price" in r3_pass_ca.columns:
    r4_fail_ca = r3_pass_ca.filter(
        r3_pass_ca["selling_price"].isNotNull() & (r3_pass_ca["selling_price"] <= 0)
    ).withColumn("_reject_reason", F.lit("R4: selling_price is zero or negative"))
    r4_pass_ca = r3_pass_ca.filter(
        r3_pass_ca["selling_price"].isNull() | (r3_pass_ca["selling_price"] > 0))
    log_dq("cars","R4_selling_price_positive","quarantine",
           r4_fail_ca.count(), total_cars,
           "Zero or negative selling price.","Check source for placeholder values.")
else:
    r4_fail_ca = r3_pass_ca.filter(F.lit(False))
    r4_pass_ca = r3_pass_ca

cars_reject = (r1_fail_ca
    .unionByName(r2_fail_ca, allowMissingColumns=True)
    .unionByName(r3_fail_ca, allowMissingColumns=True)
    .unionByName(r4_fail_ca, allowMissingColumns=True))
cars_clean  = add_audit(r4_pass_ca, "silver_cars")

write_silver(cars_clean, add_audit(cars_reject,"silver_cars_quarantine"),
             "cars","cars_quarantine")

print(f"  Clean  -> silver.cars:             {cars_clean.count():,}")
print(f"  Reject -> silver.cars_quarantine:  {cars_reject.count():,}")

In [0]:



# =============================================================================
# ENTITY 5 — POLICY
# =============================================================================
# Issues fixed (from DQ audit):
#   #25 policy_state uses 2-letter USPS codes; customers use full names -> map
#   #26 umbrella_limit = 0 for most records — ambiguous                 -> warn+flag
# Quality rules:
#   R1  policy_number (PK) not null  -> quarantine
#   R2  customer_id not null         -> quarantine
#   R3  policy_state in known set    -> quarantine
#   R4  umbrella_limit = 0           -> warn (not quarantine — ambiguous)
# =============================================================================

# COMMAND ----------
print("=" * 60)
print("POLICY — loading from Bronze")

raw_policy = spark.table("primeins.bronze.policy")
total_policy = raw_policy.count()

# ------------------------------------------------------------------
# TRANSFORMATION 1: Map 2-letter state codes -> full state names
# Required for joins between policy and customers on state field
# ------------------------------------------------------------------
STATE_MAP = {
    "IL":"Illinois",    "NC":"North Carolina", "OH":"Ohio",
    "PA":"Pennsylvania","SC":"South Carolina", "VA":"Virginia",
    "WV":"West Virginia","IN":"Indiana",       "NY":"New York",
    "CA":"California",  "TX":"Texas",          "FL":"Florida",
    "GA":"Georgia",     "MI":"Michigan",       "NJ":"New Jersey",
    "AZ":"Arizona",     "WA":"Washington",     "MA":"Massachusetts",
}

policy = raw_policy
state_col = next((c for c in ["policy_state","state"] if c in policy.columns), None)
if state_col:
    state_expr = F.upper(F.col(state_col))
    full_name_expr = F.lit(None).cast(T.StringType())
    for abbr, name in STATE_MAP.items():
        full_name_expr = F.when(state_expr == abbr, name).otherwise(full_name_expr)
    # If no match found, keep original (may already be a full name)
    full_name_expr = F.when(full_name_expr.isNull(),
                            F.initcap(F.col(state_col))).otherwise(full_name_expr)

    policy = (policy
        .withColumn("policy_state_full", full_name_expr)
        .withColumnRenamed(state_col, "policy_state_code"))

# ------------------------------------------------------------------
# TRANSFORMATION 2: Cast umbrella_limit to numeric + flag zero values
# Cannot determine if 0 = "no coverage" or "missing" — flag for business.
# ------------------------------------------------------------------
if "umbrella_limit" in policy.columns:
    policy = (policy
        .withColumn("umbrella_limit", F.col("umbrella_limit").cast(T.DoubleType()))
        .withColumn("_umbrella_limit_zero_flag", F.col("umbrella_limit") == 0))

# ------------------------------------------------------------------
# TRANSFORMATION 3: Cast numeric policy fields
# ------------------------------------------------------------------
for nc in ["policy_annual_premium","total_premium_amount",
           "capital_gains","capital_loss"]:
    if nc in policy.columns:
        policy = policy.withColumn(nc, F.col(nc).cast(T.DoubleType()))

# ------------------------------------------------------------------
# QUALITY RULES — POLICY
# ------------------------------------------------------------------

# R1: policy_number (PK) not null
pk_pol = next((c for c in ["policy_number","PolicyID","policy_id"]
               if c in policy.columns), policy.columns[0])
r1_fail_p = policy.filter(F.col(pk_pol).isNull()).withColumn(
    "_reject_reason", F.lit(f"R1: {pk_pol} is null"))
r1_pass_p = policy.filter(F.col(pk_pol).isNotNull())
log_dq("policy","R1_policy_number_not_null","quarantine",
       r1_fail_p.count(), total_policy,
       f"{pk_pol} is null.","Check source for missing policy numbers.")

# R2: customer_id not null (orphan policy cannot be owned)
cust_id_p = next((c for c in ["customer_id","CustomerID"]
                  if c in policy.columns), None)
if cust_id_p:
    r2_fail_p = r1_pass_p.filter(F.col(cust_id_p).isNull()).withColumn(
        "_reject_reason", F.lit("R2: customer_id is null — policy has no owner"))
    r2_pass_p = r1_pass_p.filter(F.col(cust_id_p).isNotNull())
    log_dq("policy","R2_customer_id_not_null","quarantine",
           r2_fail_p.count(), total_policy,
           "Policy with no customer_id cannot be linked to a policyholder.",
           "Investigate source for orphan policy records.")
else:
    r2_fail_p = r1_pass_p.filter(F.lit(False))
    r2_pass_p = r1_pass_p

# R3: policy_state_full must be in known states
if "policy_state_full" in policy.columns:
    valid_states = list(STATE_MAP.values())
    r3_fail_p = r2_pass_p.filter(
        r2_pass_p["policy_state_full"].isNull() |
        ~r2_pass_p["policy_state_full"].isin(valid_states)
    ).withColumn("_reject_reason",
        F.lit("R3: policy_state could not be mapped to a known US state"))
    r3_pass_p = r2_pass_p.filter(
        r2_pass_p["policy_state_full"].isNotNull() &
        r2_pass_p["policy_state_full"].isin(valid_states))
    log_dq("policy","R3_valid_state","quarantine",
           r3_fail_p.count(), total_policy,
           "Policy state code could not be mapped to a known US state.",
           "Add missing state codes to the lookup table in this pipeline.")
else:
    r3_fail_p = r2_pass_p.filter(F.lit(False))
    r3_pass_p = r2_pass_p

# R4: umbrella_limit = 0 -> warn only (business rule not yet defined)
if "umbrella_limit" in r3_pass_p.columns:
    zero_umbrella = r3_pass_p.filter(r3_pass_p["umbrella_limit"] == 0).count()
    log_dq("policy","R4_umbrella_limit_zero","warn",
           zero_umbrella, total_policy,
           "Most policies have umbrella_limit=0. Unclear if this means 'no coverage' or 'not populated'.",
           "Define business rule: does 0 mean no umbrella coverage? Document in data dictionary.")

policy_reject = (r1_fail_p
    .unionByName(r2_fail_p, allowMissingColumns=True)
    .unionByName(r3_fail_p, allowMissingColumns=True))
policy_clean  = add_audit(r3_pass_p, "silver_policy")

write_silver(policy_clean, add_audit(policy_reject,"silver_policy_quarantine"),
             "policy","policy_quarantine")

print(f"  Clean  -> silver.policy:             {policy_clean.count():,}")
print(f"  Reject -> silver.policy_quarantine:  {policy_reject.count():,}")


In [0]:
# =============================================================================
# FINAL: WRITE DQ ISSUES LOG TO DELTA TABLE
# primeins.silver.dq_issues_log
# Schema: run_id, logged_at, table_name, rule_name, consequence,
#         affected_count, total_count, affected_ratio, detail, suggested_fix
# =============================================================================

# COMMAND ----------
print("=" * 60)
print("Writing DQ issues log")

dq_schema = T.StructType([
    T.StructField("run_id",         T.StringType(),  False),
    T.StructField("logged_at",      T.StringType(),  True),
    T.StructField("table_name",     T.StringType(),  True),
    T.StructField("rule_name",      T.StringType(),  True),
    T.StructField("consequence",    T.StringType(),  True),
    T.StructField("affected_count", T.LongType(),    True),
    T.StructField("total_count",    T.LongType(),    True),
    T.StructField("affected_ratio", T.DoubleType(),  True),
    T.StructField("detail",         T.StringType(),  True),
    T.StructField("suggested_fix",  T.StringType(),  True),
])

dq_df = spark.createDataFrame(dq_log_rows, schema=dq_schema)

(dq_df.write.format("delta")
     .mode("append")              # append per run — full history is preserved
     .option("mergeSchema","true")
     .saveAsTable("primeins.silver.dq_issues_log"))

print(f"  DQ log entries written: {len(dq_log_rows)}")

# COMMAND ----------
# ------------------------------------------------------------------
# VERIFICATION — Silver table counts + DQ log summary
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("SILVER LAYER — FINAL VERIFICATION")
print("=" * 60)

for entity in ["customers","claims","sales","cars","policy"]:
    clean_ct = spark.table(f"primeins.silver.{entity}").count()
    quar_ct  = spark.table(f"primeins.silver.{entity}_quarantine").count()
    print(f"  {entity:<12}  clean={clean_ct:>6,}   quarantine={quar_ct:>5,}")

print("\nDQ Issues Log — this run:")
(spark.table("primeins.silver.dq_issues_log")
    .filter(F.col("run_id") == RUN_ID)
    .select("table_name","rule_name","consequence",
            "affected_count","affected_ratio","suggested_fix")
    .orderBy("table_name","rule_name")
    .show(50, truncate=70))